# Synthetic Data Generator for Subscription Finance Analytics

### Eskwelabs Innovation Fellowship Cohort 10 | Data Modeling
**Chass Miguel E. Mesina and Margrette Sofia M. Hidalgo**

June 2026

In [1]:
#Imports
import numpy as np
import pandas as pd
import datetime

np.random.seed(42)

In [2]:
#PARAMETERS

# Customer ARR Distribution (Log-Normal)
MU_ARR = 17.5       # Average scale
SIGMA2_ARR = 1.5    # Size spread

# Subscription Pricing Curve (Power-Law)
BETA_0 = 4.0        # Pricing baseline
BETA_1 = 0.5        # Power exponent

# Industry Order: [Cloud/IT, Pharma, Healthcare, Retail, Manufacturing, Construction]
INDUSTRIES = ['Cloud/IT', 'Pharma', 'Healthcare', 'Retail', 'Manufacturing', 'Construction']
INDUSTRY_MODIFIERS = np.array([1.15, 1.05, 1.00, 0.95, 0.90, 0.85])
K_INDUSTRY = np.array([100.0, 17.5, 92.0, 50.0, 17.5, 15.0])
THETA_INDUSTRY = np.array([0.5, 4.0, 0.5, 0.5, 3.0, 6.0])

# Multiplicative Contract Noise
SIGMA2_ACV = 0.05   # Pricing noise

# Telemetry & Platform Utilization AR(1) Constants
RHO = 0.75          # Persistent Memory
MU_USAGE = 0.75     # Baseline Target
SIGMA2_USAGE = 0.02 # Usage variance

# Late-Stage Contract Churn & Expansion Logistic Framework
ALPHA_0 = 1.50      # Churn Intercept
ALPHA_1 = 4.50      # Usage Sensitivity

# Section V: Latent (Omitted) Variables Baseline Settings
P_DISTRESS = 0.06   # Distress probability
P_TURNOVER = 0.15   # Turnover probability
LAMBDA_TECH_DEBT = 5.00  # Debt scale

In [3]:
#TABLE 1: CUSTOMERS

#GENERATE COMPANY NAMES
def generate_unique_company_names(industries_vector):
    roots = {
        'Cloud/IT': ['Net', 'Cyber', 'Cloud', 'Byte', 'Infra', 'Data', 'Logi', 'App', 'Sys', 'Quantum'],
        'Pharma': ['Astra', 'Bio', 'Thera', 'Pharma', 'Cura', 'Gen', 'Med', 'Vax', 'Syn', 'Helix'],
        'Healthcare': ['Apex', 'Care', 'Metro', 'Nu', 'Vanguard', 'Omni', 'Pulse', 'St. Jude', 'Beacon', 'Horizon'],
        'Retail': ['Shop', 'Nordic', 'Urban', 'Mart', 'Cart', 'Luxe', 'Velvet', 'Thrift', 'Trend', 'Bazaar'],
        'Manufacturing': ['Steel', 'Apex', 'Titan', 'Matrix', 'Kodiak', 'Global', 'Forge', 'Alloy', 'Vulcan', 'Iron'],
        'Construction': ['Build', 'Vanguard', 'Ironclad', 'Apex', 'Summit', 'Delta', 'Foundry', 'Pillar', 'Stone', 'Anchor']
    }

    modifiers = ['Vortex', 'Shield', 'Pinnacle', 'Core', 'Forge', 'Scale', 'Quantum', 'Genix', 'Nova', 'Glow',
                 'Systems', 'Labs', 'Pulse', 'Clinic', 'Vibe', 'Goods', 'Thread', 'Prime', 'Flow', 'Market',
                 'Works', 'Tooling', 'Assembly', 'Heavy', 'Mill', 'Excavation', 'Frameworks', 'Infrastructure', 'Masonry', 'Dredge']

    suffixes = ['Corp', 'Inc', 'Ltd', 'Holdings', 'Enterprise', 'Solutions', 'Group', 'Global']

    names = []
    seen_names = set()

    for ind in industries_vector:
        while True:
            root = np.random.choice(roots[ind])
            mod = np.random.choice(modifiers)
            suffix = np.random.choice(suffixes)
            sep = "" if np.random.rand() > 0.5 else " "
            candidate = f"{root}{sep}{mod} {suffix}"

            if candidate not in seen_names:
                seen_names.add(candidate)
                names.append(candidate)
                break

    return np.array(names)

#Sample scale
N_COMPANIES = 5000

customer_ids = np.arange(1001, 1001 + N_COMPANIES)

# 1. Define the categorical probability vector matching the paper
industry_weights = [0.30, 0.15, 0.20, 0.15, 0.12, 0.08]

# 2. Sample industries using the weights vector
sampled_industries = np.random.choice(INDUSTRIES, size=N_COMPANIES, p=industry_weights)

# 3. Generate company names based on the weighted industries
company_names = generate_unique_company_names(sampled_industries)

# 4. Generate company scale
company_arr = np.exp(np.random.normal(MU_ARR, np.sqrt(SIGMA2_ARR), size=N_COMPANIES))

# 5. Structural Latent Variables
epsilon_tech_debt = np.random.normal(0, 0.01, size=N_COMPANIES)
latent_tech_debt = np.clip((LAMBDA_TECH_DEBT / np.log(company_arr)) + epsilon_tech_debt, 0.0, 1.0)
latent_cashflow_distress = np.random.binomial(1, P_DISTRESS, size=N_COMPANIES)
latent_champion_turnover = np.random.binomial(1, P_TURNOVER, size=N_COMPANIES)

# Assemble DataFrame
df_customers = pd.DataFrame({
    'customer_id': customer_ids,
    'company_name': company_names,
    'industry': sampled_industries,
    'company_arr': company_arr,
    'latent_tech_debt': latent_tech_debt,
    'latent_cashflow_distress': latent_cashflow_distress,
    'latent_champion_turnover': latent_champion_turnover
})

# Display sample of the generated accounts
df_customers.head()

,customer_id,company_name,industry,company_arr,latent_tech_debt,latent_cashflow_distress,latent_champion_turnover
0,1001,TheraGenix Group,Pharma,1.377356e+08,0.287916,0,0
1,1002,IroncladScale Holdings,Construction,1.674944e+07,0.305751,0,0
2,1003,CartGoods Group,Retail,5.044642e+07,0.288444,0,0
3,1004,MetroExcavation Enterprise,Healthcare,6.339990e+07,0.270119,0,0
4,1005,Infra Nova Group,Cloud/IT,1.414945e+08,0.269588,0,0


In [4]:
df_customers['industry'].value_counts(normalize=True)

industry
Cloud/IT         0.3062
Healthcare       0.1968
Pharma           0.1492
Retail           0.1480
Manufacturing    0.1212
Construction     0.0786
Name: proportion, dtype: float64

In [5]:
#TABLE 2: PRODUCT PLANS

product_plans_data = {
    'plan_id': [1, 2],
    'plan_name': ['Growth Tier', 'Enterprise Tier'],
    'plan_seat_price': [45.00, 120.00]
}
df_product_plans = pd.DataFrame(product_plans_data)

In [6]:
#TABLE 4: USAGEEVENTS
#RUN THIS BEFORE TABLE 3

# 1. Expand arrays across 12 months to build time-series framework
months = np.arange(1, 13)

# Fix: Define sub_ids and provisioned_seats_raw based on TABLE 3's logic,
# as these are required before df_subscriptions is created in TABLE 3.
# This ensures TABLE 4 can run before TABLE 3, as indicated by the comment.

# Define sub_ids (from df_subscriptions construction in TABLE 3)
sub_ids = np.arange(5001, 5001 + N_COMPANIES)

# Define provisioned_seats_raw by replicating necessary calculations from TABLE 3
company_arr_vec = df_customers['company_arr'].values
industry_to_idx = {ind: idx for idx, ind in enumerate(INDUSTRIES)}
industry_indices = df_customers['industry'].map(industry_to_idx).values
industry_mods_vec = INDUSTRY_MODIFIERS[industry_indices]

epsilon_acv = np.random.normal(0, np.sqrt(SIGMA2_ACV), size=N_COMPANIES)
acv_raw = BETA_0 * (company_arr_vec ** BETA_1) * industry_mods_vec * np.exp(epsilon_acv)

acv_median = np.median(acv_raw)
plan_id_assignments = np.where(acv_raw >= acv_median, 2, 1)

# Use hardcoded seat prices from df_product_plans or TABLE 3 for consistency
seat_prices = np.where(plan_id_assignments == 2, 120.00, 45.00)

provisioned_seats_raw = np.floor(acv_raw / (seat_prices * 12)).astype(int)
provisioned_seats_raw = np.clip(provisioned_seats_raw, 1, None) # Enforce strict 1-seat minimum

seats_vec = provisioned_seats_raw

# Create repetitive indexes for 60,000 rows (5,000 sub_ids x 12 months)
expanded_sub_ids = np.repeat(sub_ids, 12)
expanded_months = np.tile(months, N_COMPANIES)
expanded_seats = np.repeat(seats_vec, 12)

# Fix: Replace merge with df_subscriptions. Instead, derive customer_id based on sub_id
# Assuming a direct correspondence: customer_id = df_customers['customer_id'].values[(subscription_id - 5001)]
expanded_customer_ids = df_customers['customer_id'].values[(expanded_sub_ids - 5001)]

# Merge customer latent variables into the time-series space for easy vector math
df_expanded_base = pd.DataFrame({
    'subscription_id': expanded_sub_ids,
    'month_index': expanded_months,
    'provisioned_seats': expanded_seats,
    'customer_id': expanded_customer_ids # Added customer_id directly
})

df_expanded = df_expanded_base.merge(
    df_customers[['customer_id', 'latent_tech_debt', 'latent_champion_turnover']],
    on='customer_id'
)

# Sort strictly by subscription and month to keep the time-series contiguous
df_expanded = df_expanded.sort_values(by=['subscription_id', 'month_index']).reset_index(drop=True)

# Extract working vectors
t_debt_vec = df_expanded['latent_tech_debt'].values
turnover_vec = df_expanded['latent_champion_turnover'].values
month_vec = df_expanded['month_index'].values

# Initialize empty array for utilization rate simulation
utilization_rates = np.zeros(len(df_expanded))

# 2. Iterative time-series generation across the 12-month sequence
# Initialize Month 1 using Truncated Normal properties clamped tightly between 0 and 1
m1_idx = (month_vec == 1)

# Step 1: Base month 1 initialization around MU_USAGE target minus early deployment friction
epsilon_usage_m1 = np.random.normal(0, np.sqrt(SIGMA2_USAGE), size=N_COMPANIES)
utilization_rates[m1_idx] = MU_USAGE + epsilon_usage_m1
utilization_rates[m1_idx] = np.clip(utilization_rates[m1_idx], 0.0, 1.0)

# Step 2: Propagate Months 2 through 12 sequentially to maintain AR(1) memory ties
for m in range(2, 13):
    current_m_idx = (month_vec == m)
    prior_m_idx = (month_vec == (m - 1))

    # Structural Shocks & Penalties calculation for month m
    onboarding_penalty = np.where(m <= 3, t_debt_vec[current_m_idx] / 20.0, 0.0)
    turnover_shock = np.where(m >= 6, 0.45 * turnover_vec[current_m_idx], 0.0)

    # Draw random idiosyncratic usage innovation noise
    epsilon_usage = np.random.normal(0, np.sqrt(SIGMA2_USAGE), size=N_COMPANIES)

    # Run Mean-Reverting AR(1) formula: mu + rho*(prev - mu) - penalties - shocks + noise
    prev_rates = utilization_rates[prior_m_idx]
    rates_m = MU_USAGE + RHO * (prev_rates - MU_USAGE) - onboarding_penalty - turnover_shock + epsilon_usage

    # Enforce standard measurement boundaries [0.0, 1.0]
    utilization_rates[current_m_idx] = np.clip(rates_m, 0.0, 1.0)

# Add utilization rates to final DataFrame
df_expanded['utilization_rate'] = utilization_rates

# 3. Derive absolute workforce logins (active_users) locked to seat constraints
df_expanded['active_users'] = np.round(df_expanded['utilization_rate'] * df_expanded['provisioned_seats']).astype(int)

# Construct final clean UsageEvents Table
df_usage_events = df_expanded[[
    'subscription_id', 'month_index', 'active_users', 'utilization_rate'
]].copy()
df_usage_events.insert(0, 'event_id', np.arange(100001, 100001 + len(df_usage_events)))

# Check the baseline logs for subscription 5001 across its 12 months
df_usage_events.head(12)

,event_id,subscription_id,month_index,active_users,utilization_rate
0,100001,5001,1,30,0.671333
1,100002,5001,2,32,0.715703
2,100003,5001,3,34,0.753693
3,100004,5001,4,42,0.926999
4,100005,5001,5,37,0.813334
5,100006,5001,6,30,0.677740
6,100007,5001,7,27,0.603771
7,100008,5001,8,33,0.729966
8,100009,5001,9,34,0.760068
9,100010,5001,10,40,0.887444


In [7]:
#TABLE 3: SUBSCRIPTIONS

# 1. Map industry strings to index values to align with parameter vectors
industry_to_idx = {ind: idx for idx, ind in enumerate(INDUSTRIES)}
industry_indices = df_customers['industry'].map(industry_to_idx).values

# Extract vectors for vectorized computation
company_arr_vec = df_customers['company_arr'].values
industry_mods_vec = INDUSTRY_MODIFIERS[industry_indices]

# Reuse the ACV draw, plan tier and seat counts already computed in the UsageEvents cell (Table 4),
# so every table describes the same contracts. Re-drawing epsilon_acv here made the seat counts
# in Subscriptions disagree with the seats used to simulate UsageEvents.
provisioned_seats = provisioned_seats_raw

# Finalize smooth ACV to maintain absolute mathematical parity with seat totals
acv = provisioned_seats * seat_prices * 12

# Billing Cycle Assignment: Enterprise plans get Annual Upfront; Growth plans pay Quarterly
billing_cycles = np.where(plan_id_assignments == 2, 'Annual Upfront', 'Quarterly')

# Chronological Dates: Setup a unified activation matrix
start_date = datetime.date(2025, 1, 1)
end_date = start_date + datetime.timedelta(days=365)

# 2. Extract Q4 Mean Utilization Rate from df_usage_events to process late-stage outcomes
q4_logs = df_usage_events[df_usage_events['month_index'].isin([9, 10, 11, 12])]
q4_mean_usage = q4_logs.groupby('subscription_id')['utilization_rate'].mean().reset_index()
q4_mean_usage.columns = ['subscription_id', 'u_bar_q4']
u_bar_vec = q4_mean_usage['u_bar_q4'].values

# Calculate Logistic Churn Probability
log_odds = -(ALPHA_0 - ALPHA_1 * u_bar_vec)
p_churn = 1.0 / (1.0 + np.exp(log_odds))

# Execute stochastic Bernoulli trial for contract cancellation
churn_trial = np.random.binomial(1, p_churn)

# Apply Sequential Multi-Class Sorting Threshold Rules
final_statuses = np.empty(N_COMPANIES, dtype=object)
final_statuses[churn_trial == 1] = 'Churned'

survived_and_expanded = (churn_trial == 0) & (u_bar_vec >= 0.90)
final_statuses[survived_and_expanded] = 'Expanded'

survived_and_renewed = (churn_trial == 0) & (u_bar_vec < 0.90)
final_statuses[survived_and_renewed] = 'Renewed'

# 3. Assemble Final Unified Subscriptions DataFrame
df_subscriptions = pd.DataFrame({
    'subscription_id': np.arange(5001, 5001 + N_COMPANIES),
    'customer_id': df_customers['customer_id'],
    'plan_id': plan_id_assignments,
    'acv': acv,
    'provisioned_seats': provisioned_seats,
    'billing_cycle': billing_cycles,
    'start_date': start_date,
    'end_date': end_date,
    'u_bar_q4': u_bar_vec,
    'contract_status': final_statuses
})

# Display sample of the completed commercial ledger
df_subscriptions.head()

,subscription_id,customer_id,plan_id,acv,provisioned_seats,billing_cycle,start_date,end_date,u_bar_q4,contract_status
0,5001,1001,2,64800.0,45,Annual Upfront,2025-01-01,2026-01-01,0.673663,Churned
1,5002,1002,1,18900.0,35,Quarterly,2025-01-01,2026-01-01,0.792348,Renewed
2,5003,1003,2,28800.0,20,Annual Upfront,2025-01-01,2026-01-01,0.937462,Expanded
3,5004,1004,2,28800.0,20,Annual Upfront,2025-01-01,2026-01-01,0.820895,Renewed
4,5005,1005,2,57600.0,40,Annual Upfront,2025-01-01,2026-01-01,0.799452,Renewed


In [8]:
#TABLE 5 & 6: INVOICES & PAYMENTS

# 1. Expand subscription contracts into their relational invoice schedules
invoice_records = []
payment_records = []

invoice_id_counter = 200001
payment_id_counter = 300001

# Fetch underlying structural customer arrays via a joined temporary dictionary for fast lookups
customer_lookup = df_customers.set_index('customer_id')[['industry', 'latent_cashflow_distress']].to_dict('index')

for _, sub in df_subscriptions.iterrows():
    sub_id = sub['subscription_id']
    cust_id = sub['customer_id']
    acv_val = sub['acv']
    cycle = sub['billing_cycle']

    # Extract sector metadata and latent corporate cashflow failure variable
    cust_meta = customer_lookup[cust_id]
    ind_name = cust_meta['industry']
    c_i = cust_meta['latent_cashflow_distress']
    ind_idx = industry_to_idx[ind_name]

    # Extract structural Gamma shape and scale factors for this specific sector
    k_param = K_INDUSTRY[ind_idx]
    theta_param = THETA_INDUSTRY[ind_idx]

    # Assign chronological schedules and billing splits
    if cycle == 'Annual Upfront':
        billing_intervals = [1]  # Billed only once at Month 1 (Start of contract)
        split_factor = 1.0
    else:
        billing_intervals = [1, 4, 7, 10]  # Billed every quarter
        split_factor = 0.25

    amount_due = acv_val * split_factor

    for m_inc in billing_intervals:
        # Compute dynamic date stamps based on billing month intervals
        inv_date = datetime.date(2025, m_inc, 1)

        # --- PROCESS TABLE 5: INVOICES ---
        inv_row = {
            'invoice_id': invoice_id_counter,
            'subscription_id': sub_id,
            'invoice_date': inv_date,
            'amount_due': amount_due
        }
        invoice_records.append(inv_row)

        # --- PROCESS TABLE 6: PAYMENTS ---
        # Draw standard transactional clearing time from the sector's Gamma profile
        base_days_to_pay = np.random.gamma(shape=k_param, scale=theta_param)

        # Incorporate the hidden financial distress multiplier trap
        # exp(1.5 * 1) = ~4.48x multiplier on collection lag if distressed
        final_days_to_pay = int(np.round(base_days_to_pay * np.exp(1.5 * c_i)))

        # Enforce strict zero-bound for instantaneous automated clearances
        final_days_to_pay = max(0, final_days_to_pay)

        # Construct actual corporate collection resolution dates
        pay_date = inv_date + datetime.timedelta(days=final_days_to_pay)

        # Evaluate operational status based on explicit accounting aging rules
        if final_days_to_pay > 90:
            status = 'Bad_Debt'
        elif final_days_to_pay > 60:
            status = 'In_Collections'
        else:
            status = 'Paid'

        pay_row = {
            'payment_id': payment_id_counter,
            'invoice_id': invoice_id_counter,
            'days_to_pay': final_days_to_pay,
            'payment_date': pay_date,
            'payment_status': status
        }
        payment_records.append(pay_row)

        invoice_id_counter += 1
        payment_id_counter += 1

# Compile collections data matrices into formal Pandas frames
df_invoices = pd.DataFrame(invoice_records)
df_payments = pd.DataFrame(payment_records)

# Check samples from both sub-ledgers
df_invoices.head()

,invoice_id,subscription_id,invoice_date,amount_due
0,200001,5001,2025-01-01,64800.0
1,200002,5002,2025-01-01,4725.0
2,200003,5002,2025-04-01,4725.0
3,200004,5002,2025-07-01,4725.0
4,200005,5002,2025-10-01,4725.0


In [9]:
#TABLE 7: REVENUESCHEDULES

revenue_records = []
schedule_id_counter = 400001

for _, sub in df_subscriptions.iterrows():
    sub_id = sub['subscription_id']
    acv_val = sub['acv']

    # Calculate recognized monthly revenue (ACV / 12)
    recognized_mrr = acv_val / 12.0

    # Initialize the starting deferred revenue liability balance
    deferred_balance = acv_val

    for month in range(1, 13):
        # Decrement the remaining deferred balance by the recognized portion
        deferred_balance -= recognized_mrr

        rev_row = {
            'schedule_id': schedule_id_counter,
            'subscription_id': sub_id,
            'fiscal_month': month,
            'recognized_mrr': np.round(recognized_mrr, 2),
            'deferred_revenue_balance': max(0.0, np.round(deferred_balance, 2))
        }
        revenue_records.append(rev_row)
        schedule_id_counter += 1

# Compile the corporate accrual accounting sub-ledger
df_revenue_schedules = pd.DataFrame(revenue_records)

# Verify the final ledger output for subscription 5001
df_revenue_schedules.head(12)

,schedule_id,subscription_id,fiscal_month,recognized_mrr,deferred_revenue_balance
0,400001,5001,1,5400.0,59400.0
1,400002,5001,2,5400.0,54000.0
2,400003,5001,3,5400.0,48600.0
3,400004,5001,4,5400.0,43200.0
4,400005,5001,5,5400.0,37800.0
5,400006,5001,6,5400.0,32400.0
6,400007,5001,7,5400.0,27000.0
7,400008,5001,8,5400.0,21600.0
8,400009,5001,9,5400.0,16200.0
9,400010,5001,10,5400.0,10800.0


In [10]:
#CSV EXPORT

df_customers_student = df_customers.drop(columns=['latent_tech_debt', 'latent_cashflow_distress', 'latent_champion_turnover'])
df_subscriptions_student = df_subscriptions.drop(columns=['u_bar_q4'])

df_customers_student.to_csv('customers.csv', index=False)
df_product_plans.to_csv('product_plans.csv', index=False)
df_subscriptions_student.to_csv('subscriptions.csv', index=False)
df_usage_events.to_csv('usage_events.csv', index=False)
df_invoices.to_csv('invoices.csv', index=False)
df_payments.to_csv('payments.csv', index=False)
df_revenue_schedules.to_csv('revenue_schedules.csv', index=False)

